# Model Serving with FastAPI — Exercises

> 📘 **Python Mastery** · Module 18 — MLOps · Lesson 4/6
>
> Practice serving trained models through validated HTTP APIs using FastAPI and TestClient.

## Setup

In [ ]:
# Required imports for all exercises
from fastapi import FastAPI, HTTPException, Query
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field, validator
import numpy as np
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

---
## Part 1: Conceptual Questions

Answer these questions to test your understanding of model serving concepts.

### Exercise 1.1: Batch vs Real-Time

**Question:** For each scenario below, decide whether batch scoring or a real-time API is more appropriate, and explain why:

1. A credit card company wants to detect fraudulent transactions at the moment of purchase
2. A marketing team wants to score all customers for churn risk to plan next month's retention campaign
3. A ride-sharing app needs to estimate arrival times when users request a ride
4. An analytics team wants to re-score all historical orders with an updated recommendation model

Write your answers in the cell below:

**Your Answer:**

1. 
2. 
3. 
4. 

### Exercise 1.2: Validation Strategy

**Question:** Why does FastAPI return HTTP 422 for invalid requests instead of HTTP 400 (Bad Request) or HTTP 500 (Internal Server Error)? What does this communicate to API clients?

**Your Answer:**



### Exercise 1.3: Model Loading Strategy

**Question:** Explain the difference between these two approaches to model loading. Which is correct and why?

**Approach A:**
```python
import joblib
from fastapi import FastAPI

app = FastAPI()
model = joblib.load("model.pkl")  # Module level

@app.post("/predict")
def predict(features: Features):
    return {"prediction": model.predict([features.values])}
```

**Approach B:**
```python
import joblib
from fastapi import FastAPI

app = FastAPI()

@app.post("/predict")
def predict(features: Features):
    model = joblib.load("model.pkl")  # Inside endpoint
    return {"prediction": model.predict([features.values])}
```

**Your Answer:**



### Exercise 1.4: NumPy Type Conversion

**Question:** Why must we convert NumPy types (like `np.float32`, `np.bool_`) to Python built-in types (`float`, `bool`) before returning them in a FastAPI response? What error occurs if we don't?

**Your Answer:**



---
## Part 2: Coding Exercises

Build FastAPI applications with proper validation and testing.

### Exercise 2.1: Basic Routes with Path and Query Parameters

Create a FastAPI app with two endpoints:
1. `GET /models/{model_id}` - Returns model information for the given model_id (string)
2. `GET /models/{model_id}/metrics` - Returns model metrics with optional query parameter `metric_type` (default: "accuracy")

Test both endpoints using TestClient. The `/models/{model_id}` endpoint should return:
```json
{"model_id": "...", "status": "active", "version": "1.0.0"}
```

The `/models/{model_id}/metrics` endpoint should return:
```json
{"model_id": "...", "metric_type": "...", "value": 0.95}
```

In [ ]:
# Your code here


### Exercise 2.2: Pydantic Request Validation

Create a Pydantic model `HousingFeatures` for a house price prediction API with these constraints:
- `bedrooms`: integer, between 1 and 10
- `bathrooms`: float, between 0.5 and 8.0
- `sqft`: float, between 100 and 10000
- `year_built`: integer, between 1800 and 2030

Create a FastAPI app with a `POST /validate` endpoint that accepts `HousingFeatures` and returns the validated data.

Write tests that:
1. Verify a valid payload returns 200
2. Verify invalid `bedrooms` (0 or 20) returns 422
3. Verify invalid `sqft` (50 or 50000) returns 422

In [ ]:
# Your code here


### Exercise 2.3: Serving a Classification Model

Train a simple logistic regression model to predict if a student passes (1) or fails (0) based on hours studied and attendance rate.

Training data:
```python
X_train = [[2, 0.6], [4, 0.8], [1, 0.5], [6, 0.9], [3, 0.7], [5, 0.85]]
y_train = [0, 1, 0, 1, 0, 1]
```

Create:
1. A Pydantic model `StudentFeatures` with:
   - `hours_studied`: float between 0 and 80
   - `attendance`: float between 0 and 1
2. A FastAPI app with `POST /predict` that returns:
   - `probability`: rounded to 4 decimals
   - `prediction`: boolean (True if probability >= 0.5)
   - `model_version`: "1.0.0"
3. Tests verifying the response structure and types

In [ ]:
# Your code here


### Exercise 2.4: Health Check Endpoint

Create a FastAPI app with three endpoints:
1. `GET /health` - Returns `{"status": "ok", "model_version": "1.0.0", "model_loaded": true}`
2. `GET /version` - Returns `{"version": "1.0.0", "framework": "sklearn"}`
3. `POST /predict` - A dummy prediction endpoint

Write tests that verify:
1. `/health` returns 200 with status "ok"
2. `/version` contains the expected keys
3. All endpoints respond within reasonable structure

In [ ]:
# Your code here


### Exercise 2.5: Multi-Feature Regression API

Create a FastAPI app that serves a linear regression model predicting house prices.

Training data:
```python
X_train = [[1500, 3, 2], [2000, 4, 3], [1200, 2, 1], [2500, 4, 3], [1800, 3, 2]]
y_train = [300000, 400000, 250000, 500000, 350000]
# Features: [sqft, bedrooms, bathrooms]
```

Requirements:
1. Pydantic model with appropriate constraints
2. `POST /predict` returns:
   - `predicted_price`: float rounded to 2 decimals
   - `input_features`: dict of the input
   - `model_version`: "1.0.0"
3. Proper NumPy type conversion
4. Tests for valid input and boundary violations

In [ ]:
# Your code here


### Exercise 2.6: Error Handling with Custom Validation

Create a Pydantic model `CreditCardFeatures` with a custom validator that ensures the `credit_limit` is greater than `current_balance`.

Fields:
- `credit_limit`: float, between 500 and 50000
- `current_balance`: float, between 0 and 50000
- `payment_history_months`: int, between 1 and 120

Custom validation: `current_balance` must be <= `credit_limit`

Create an endpoint and tests that verify:
1. Valid data passes
2. `current_balance > credit_limit` is rejected
3. Out-of-range values are rejected

In [ ]:
# Your code here


---
## Part 3: Challenge Problems

More complex scenarios combining multiple concepts.

### Exercise 3.1: Multi-Model API with Model Selection

Create a FastAPI app that serves TWO different models:
1. A logistic regression classifier
2. A random forest classifier

Both models should be trained on the same student pass/fail dataset.

Create endpoints:
1. `POST /predict/logistic` - Uses logistic regression
2. `POST /predict/random-forest` - Uses random forest
3. `GET /models` - Lists available models with their versions

Each prediction endpoint should return:
- `probability`: float
- `prediction`: bool
- `model_type`: string ("logistic_regression" or "random_forest")
- `model_version`: "1.0.0"

Write comprehensive contract tests for both endpoints.

In [ ]:
# Your code here


### Exercise 3.2: Batch Prediction Endpoint

Create a FastAPI app with a `POST /predict/batch` endpoint that accepts a list of feature dictionaries and returns predictions for all of them.

Requirements:
1. Pydantic model `StudentFeatures` (hours_studied, attendance)
2. Request model `BatchRequest` with a field `samples` that is a list of `StudentFeatures`
3. The list must contain between 1 and 100 samples (use Pydantic's `Field` with `min_items` and `max_items`)
4. Response includes:
   - `predictions`: list of dicts with individual predictions
   - `count`: number of predictions
   - `model_version`: "1.0.0"

Write tests for:
1. Single sample
2. Multiple valid samples
3. Empty list rejection
4. Over-limit list rejection (>100 samples)

In [ ]:
# Your code here


### Exercise 3.3: Full Production-Ready API

Build a complete, production-ready model serving API with all best practices:

**Requirements:**

1. **Model**: Train a scikit-learn Pipeline with StandardScaler + LogisticRegression on student data

2. **Endpoints**:
   - `GET /` - Root endpoint with API info and links to `/docs` and `/health`
   - `GET /health` - Health check with model status
   - `GET /version` - Model and API version info
   - `POST /predict` - Main prediction endpoint
   - `GET /models/metadata` - Returns model training metadata (feature names, training date, etc.)

3. **Validation**:
   - Pydantic models with comprehensive constraints
   - Custom validators where needed
   - Proper error messages

4. **Response Format**:
   - All NumPy types converted to Python types
   - Model version in every prediction response
   - Echo input features in prediction response for auditability
   - Include confidence scores

5. **Testing**:
   - Contract tests for all endpoints
   - Happy path tests
   - Boundary value tests
   - Invalid input tests
   - Response structure validation

**Bonus**: Add a `GET /models/explain` endpoint that returns feature importance or model coefficients.

In [ ]:
# Your code here


---
## Reflection

After completing these exercises:

1. What are the three most important validations you should always include in a model serving API?
2. Why is TestClient superior to starting a real server for testing?
3. How would you handle versioning if you needed to deploy a breaking change to your model API?

**Your Reflection:**

